# 投稿動画数の分析

旧11後半の移植。動画マスタ（`01_collect/video_master` の出力、なければAPIから取得）を使い、
年間投稿本数・月別Shorts比率・投稿カレンダーヒートマップを描く。

In [ ]:
#@title 🔧 セットアップ（パッケージinstall + Driveマウント）
GITHUB_OWNER = "asmrt"  # ← GitHubユーザー名に変更

from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q "git+https://{token}@github.com/{GITHUB_OWNER}/unofficial_sixfonia_analytics.git"
drive.mount("/content/drive")

In [ ]:
#@title ⚙️ 設定と動画マスタのロード
CHANNEL = "hima72"  #@param ["hima72", "sixfonia", "kosame", "illuma", "mikoto", "suchi", "lan"]

import pandas as pd
from sixfonia_analytics import auth, collect, config

master_path = config.channel_dir(CHANNEL) / f"{CHANNEL}_video_master.csv"
if master_path.exists():
    df = pd.read_csv(master_path)
    print(f"保存済みマスタを使用: {master_path}")
else:
    print("保存済みマスタがないためAPIから取得します...")
    youtube = auth.build_youtube()
    df = pd.DataFrame(collect.fetch_video_master(youtube, config.channel_id_of(CHANNEL)))

df["published_at"] = pd.to_datetime(df["published_at"])
df = df.sort_values("published_at").reset_index(drop=True)
print(f"取得件数: {len(df)}")
df.tail(3)

In [ ]:
#@title 📊 年間投稿本数
import matplotlib.pyplot as plt
import seaborn as sns
from sixfonia_analytics import plots

plots.setup_japanese_font()

yearly = df["published_at"].dt.year.value_counts().sort_index()
plt.figure(figsize=(12, 6))
sns.barplot(x=yearly.index, y=yearly.values, palette="viridis")
plt.title(f"{CHANNEL} 年間投稿本数")
plt.xlabel("年")
plt.ylabel("投稿本数")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
#@title 📊 月別: 通常動画 vs Shorts（積み上げ）
monthly = (
    df.assign(ym=df["published_at"].dt.to_period("M"))
    .groupby(["ym", "is_short"]).size().unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(15, 7))
monthly.plot(kind="bar", stacked=True, ax=ax, color=["skyblue", "salmon"])
ax.set_title(f"{CHANNEL} 年月ごとの投稿本数（通常動画 / Shorts）")
ax.set_xlabel("年月")
ax.set_ylabel("投稿本数")
ax.legend(["通常の動画", "ショート動画"], title="動画の種類")
plt.xticks(rotation=90)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

display(monthly.tail(12))

In [ ]:
#@title 🗓️ 投稿カレンダーヒートマップ（期間指定）
TARGET_YEAR = 2026  #@param {type:"integer"}
START_MONTH = 1  #@param {type:"integer"}
END_MONTH = 6  #@param {type:"integer"}

import numpy as np

daily = df.groupby(df["published_at"].dt.date).size().reset_index(name="count")
daily["published_at"] = pd.to_datetime(daily["published_at"])

start = pd.Timestamp(f"{TARGET_YEAR}-{START_MONTH:02d}-01")
end = pd.Timestamp(f"{TARGET_YEAR}-{END_MONTH:02d}-01") + pd.offsets.MonthEnd(0)
full = pd.DataFrame({"published_at": pd.date_range(start, end, freq="D")})
merged = full.merge(daily, on="published_at", how="left").fillna(0)

merged["dow"] = merged["published_at"].dt.dayofweek
first_week = pd.Timestamp(f"{TARGET_YEAR}-01-01").isocalendar().week
week = merged["published_at"].dt.isocalendar().week.astype(int)
if pd.Timestamp(f"{TARGET_YEAR}-01-01").dayofweek != 0:
    week = week - first_week + 1
    week[week <= 0] = 53
merged["week"] = week

heat = merged.pivot_table(index="dow", columns="week", values="count", fill_value=np.nan)

plt.figure(figsize=(20, 8))
ax = sns.heatmap(heat, linewidths=0.5, linecolor="lightgray", cmap="YlGnBu",
                 cbar_kws={"label": "投稿数"}, square=True, annot=True,
                 fmt="g", annot_kws={"fontsize": 8})

month_labels, month_pos = [], []
for month in range(START_MONTH, END_MONTH + 1):
    first = pd.Timestamp(f"{TARGET_YEAR}-{month:02d}-01")
    w = first.isocalendar().week
    if first.dayofweek != 0:
        w = w - first_week + 1
        if w <= 0:
            w = 53
    month_labels.append(f"{month}月")
    month_pos.append(w - 0.5)
ax.set_xticks(month_pos)
ax.set_xticklabels(month_labels, rotation=0)
ax.set_yticklabels(["月", "火", "水", "木", "金", "土", "日"], rotation=0)
ax.set_title(f"{CHANNEL} {TARGET_YEAR}年{START_MONTH}月〜{END_MONTH}月の投稿ヒートマップ", fontsize=16)
ax.set_xlabel("週")
ax.set_ylabel("曜日")
plt.tight_layout()
plt.show()